In [4]:
import numpy as np

days = np.arange(1, 6)
et0_forecast = np.array([4.2, 4.5, 4.8, 5.0, 3.8])
rain_forecast = np.array([0.0, 0.0, 0.0, 0.0, 12.0])
Kc = 0.85
etc_forecast = et0_forecast * Kc

FIELD_CAPACITY = 35.0
WILTING_POINT = 15.0
CRITICAL_POINT = 23.0
INITIAL_MOISTURE = 20.0
ROOT_ZONE_DEPTH = 400.0
MAX_DAILY_IRR = 15.0

def moisture_to_mm(pct):
    return (pct / 100.0) * ROOT_ZONE_DEPTH

def mm_to_moisture(mm):
    return (mm / ROOT_ZONE_DEPTH) * 100.0

def evaluate_fitness(schedule, w1=1.0, w2=5.0):
    total_water_used = np.sum(schedule)
    total_stress_penalty = 0.0

    current_water_mm = moisture_to_mm(INITIAL_MOISTURE)
    fc_mm = moisture_to_mm(FIELD_CAPACITY)
    wp_mm = moisture_to_mm(WILTING_POINT)
    critical_mm = moisture_to_mm(CRITICAL_POINT)

    for t in range(len(schedule)):
        current_water_mm += schedule[t] + rain_forecast[t]
        current_water_mm -= etc_forecast[t]

        if current_water_mm > fc_mm:
            current_water_mm = fc_mm

        if current_water_mm < 0:
            current_water_mm = 0

        if current_water_mm < critical_mm:
            total_stress_penalty += (critical_mm - current_water_mm) ** 2

        if current_water_mm <= wp_mm:
            total_stress_penalty += 5000.0

    fitness = (w1 * total_water_used) + (w2 * total_stress_penalty)
    return fitness

def run_irrigation_pso(swarm_size=40, max_iter=100):
    dim = len(days)
    w = 0.8
    c1 = 1.8
    c2 = 1.8

    X = np.random.uniform(0.0, 10.0, (swarm_size, dim))
    V = np.random.uniform(-2.0, 2.0, (swarm_size, dim))

    pbest = np.copy(X)
    pbest_fitness = np.array([evaluate_fitness(p) for p in pbest])

    gbest_idx = np.argmin(pbest_fitness)
    gbest = np.copy(pbest[gbest_idx])
    gbest_fitness = pbest_fitness[gbest_idx]

    for it in range(max_iter):
        for i in range(swarm_size):
            r1, r2 = np.random.rand(dim), np.random.rand(dim)

            V[i] = (w * V[i]) + (c1 * r1 * (pbest[i] - X[i])) + (c2 * r2 * (gbest - X[i]))
            V[i] = np.clip(V[i], -3.0, 3.0)

            X[i] = X[i] + V[i]
            X[i] = np.clip(X[i], 0.0, MAX_DAILY_IRR)

            current_fit = evaluate_fitness(X[i])

            if current_fit < pbest_fitness[i]:
                pbest[i] = np.copy(X[i])
                pbest_fitness[i] = current_fit

                if current_fit < gbest_fitness:
                    gbest = np.copy(X[i])
                    gbest_fitness = current_fit

        w *= 0.98

    return gbest, gbest_fitness

np.random.seed(42)
optimal_schedule, final_cost = run_irrigation_pso()

print(" OPTIMAL 5-DAY IRRIGATION SCHEDULE ")
for d in range(len(optimal_schedule)):
    print(f"Day {d+1:02d}: Planned Irrigation = {optimal_schedule[d]:5.2f} mm | Forecasted Rain = {rain_forecast[d]:5.2f} mm | Crop Use (ETc) = {etc_forecast[d]:5.2f} mm")

print(f"Total Applied Irrigation Water: {np.sum(optimal_schedule):.2f} mm")

 OPTIMAL 5-DAY IRRIGATION SCHEDULE 
Day 01: Planned Irrigation = 15.00 mm | Forecasted Rain =  0.00 mm | Crop Use (ETc) =  3.57 mm
Day 02: Planned Irrigation =  6.36 mm | Forecasted Rain =  0.00 mm | Crop Use (ETc) =  3.82 mm
Day 03: Planned Irrigation =  6.27 mm | Forecasted Rain =  0.00 mm | Crop Use (ETc) =  4.08 mm
Day 04: Planned Irrigation =  0.00 mm | Forecasted Rain =  0.00 mm | Crop Use (ETc) =  4.25 mm
Day 05: Planned Irrigation =  0.00 mm | Forecasted Rain = 12.00 mm | Crop Use (ETc) =  3.23 mm
Total Applied Irrigation Water: 27.62 mm
